# Toxicity Fairness Benchmark — Analysis

This notebook loads pre-computed benchmark results (from `scripts/run_benchmark.py`)
and produces the full analysis: confusion matrices, fairness metric comparisons,
and statistical significance tests.

**Prerequisites:** Run the benchmark first:
```bash
python scripts/run_benchmark.py --dataset hatexplain --sample 500 --models perspective gemini claude
```

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.metrics import confusion_matrix

from toxicity_fairness.metrics.fairness import (
    group_stats,
    fairness_report,
    equalized_odds_gap,
    demographic_parity_gap,
    accuracy_gap,
)

## 1. Load results

In [ ]:
df = pd.read_parquet('../results/raw_results.parquet')
print(f'Rows: {len(df):,}')
print(f'Models: {df["model"].unique()}')
print(f'Protected attributes: {df["protected_attribute"].unique()}')
df.head()

## 2. Overall fairness report

In [ ]:
report = fairness_report(df)
report.style.format({
    'overall_accuracy': '{:.1%}',
    'accuracy_gap': '{:.1%}',
    'dp_gap': '{:.1%}',
    'tpr_gap': '{:.1%}',
    'fpr_gap': '{:.1%}',
}).background_gradient(subset=['accuracy_gap', 'dp_gap', 'tpr_gap'], cmap='RdYlGn_r')

## 3. Accuracy by subgroup

In [ ]:
gender_df = df[df['protected_attribute'] == 'Gender']

rows = []
for model, mdf in gender_df.groupby('model'):
    stats = group_stats(mdf)
    for grp, row in stats.iterrows():
        rows.append({'model': model, 'group': grp,
                     'accuracy': row['accuracy'],
                     'ci_lo': row['acc_ci_lo'], 'ci_hi': row['acc_ci_hi']})

bar_df = pd.DataFrame(rows)
fig = px.bar(
    bar_df, x='group', y='accuracy', color='model', barmode='group',
    error_y=bar_df['ci_hi'] - bar_df['accuracy'],
    error_y_minus=bar_df['accuracy'] - bar_df['ci_lo'],
    title='Accuracy by gender subgroup (95% CI)',
)
fig.update_layout(yaxis_tickformat='.0%', yaxis_range=[0, 1])
fig.show()

## 4. Equalized odds plot

In [ ]:
eo_rows = []
for model, mdf in gender_df.groupby('model'):
    stats = group_stats(mdf)
    for grp, row in stats.iterrows():
        eo_rows.append({'model': model.split('/')[-1], 'group': grp,
                        'tpr': row['tpr'], 'fpr': row['fpr'], 'n': row['n']})

eo_df = pd.DataFrame(eo_rows)
fig = px.scatter(
    eo_df, x='fpr', y='tpr', color='model', text='group', size='n',
    title='Equalized odds: TPR vs FPR by gender subgroup',
)
fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1,
              line=dict(color='gray', dash='dash'))
fig.update_traces(textposition='top center')
fig.show()

## 5. Key findings

*(Fill this in after running the notebook with real results)*

- **Finding 1:** ...
- **Finding 2:** ...

## 6. Limitations

- Sample size limits statistical power for subgroup analysis
- Protected attribute labels in source datasets are proxies, not ground truth
- LLM-based models (Gemini, Claude) are prompt-sensitive — see `docs/prompt_design.md`
- Results reflect API behavior at time of evaluation